In [0]:
%pip install "psycopg[binary,pool]" databricks-sdk --upgrade
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
lakebase_type=dbutils.widgets.get("lakebase_type")

lakebase_type

'provisioned'

### Config + token rotation helper

In [0]:
# -----------------------------
# CHOOSE LAKEBASE TYPE
# -----------------------------
# "provisioned"  → uses w.database.generate_database_credential(...)
# "autoscaling"  → uses w.postgres.generate_database_credential(...)

# -----------------------------
# COMMON POSTGRES SETTINGS
# -----------------------------
PG_PORT = 5432
PG_USER = "dipankar.kushari@databricks.com"    # must have a Postgres OAuth role

# -----------------------------
# PROVISIONED-ONLY SETTINGS
# -----------------------------
PROVISIONED_INSTANCE_NAME = "gap-online-feature-store"  # from Provisioned UI

# -----------------------------
# AUTOSCALING-ONLY SETTINGS
# -----------------------------
# e.g. "projects/<project-id>/branches/<branch-id>/endpoints/<endpoint-id>"
AUTOSCALING_ENDPOINT = "projects/dkushari-new-aslb/branches/production/endpoints/primary"

# -----------------------------
# TYPE-DEPENDENT HOST / DB
# -----------------------------
if lakebase_type == "provisioned":
    PG_HOST = "instance-f4a10c0a-8909-4f23-a804-41dae0fa3821.database.azuredatabricks.net"
    PG_DB   = "dkushari_uc"
elif lakebase_type == "autoscaling":
    PG_HOST = "ep-patient-firefly-e1f9hixo.database.eastus2.azuredatabricks.net"
    PG_DB   = "databricks_postgres"

In [0]:
import time
from databricks.sdk import WorkspaceClient

# ---- LAKEBASE CONNECTION CONFIG ----
# For Provisioned Lakebase (your host will look like instance-...database.azuredatabricks.net)

# ---- TOKEN ROTATION SETTINGS ----
TOKEN_TTL_SEC = 3600          # OAuth tokens are 1 hour
REFRESH_MARGIN_SEC = 300      # refresh 5 minutes before expiry

w = WorkspaceClient()

# Cell: cleanly tear down current pool
try:
    pool.close()       # stop handing out new connections
except NameError:
    pass  # pool not defined yet, that's fine

# Reset token cache so we mint a fresh token for the new type
_pg_token = None
_pg_token_last_refresh = 0.0

_pg_token = None
_pg_token_last_refresh = 0.0

def _mint_token_provisioned() -> str:
    cred = w.database.generate_database_credential(
        request_id=str(time.time()),
        instance_names=[PROVISIONED_INSTANCE_NAME],
    )
    return cred.token

def _mint_token_autoscaling() -> str:
    cred = w.postgres.generate_database_credential(
        endpoint=AUTOSCALING_ENDPOINT,
    )
    return cred.token

def get_pg_oauth_token() -> str:
    """Return a valid Lakebase OAuth token, refreshing before expiry."""
    global _pg_token, _pg_token_last_refresh

    now = time.time()
    needs_refresh = (
        _pg_token is None
        or now - _pg_token_last_refresh > (TOKEN_TTL_SEC - REFRESH_MARGIN_SEC)
    )

    if needs_refresh:
        if lakebase_type == "provisioned":
            _pg_token = _mint_token_provisioned()
        elif lakebase_type == "autoscaling":
            _pg_token = _mint_token_autoscaling()
        else:
            raise ValueError(f"Unknown LAKEBASE_TYPE: {lakebase_type}")
        _pg_token_last_refresh = now

    return _pg_token

### psycopg3 pool with automatic token refresh

In [0]:
import psycopg
from psycopg_pool import ConnectionPool

class OAuthConnection(psycopg.Connection):
    """Inject a fresh Lakebase OAuth token for each new physical connection."""

    @classmethod
    def connect(cls, conninfo: str = "", **kwargs):
        kwargs["password"] = get_pg_oauth_token()
        return super().connect(conninfo, **kwargs)

CONNINFO = (
    f"dbname={PG_DB} user={PG_USER} host={PG_HOST} "
    f"port={PG_PORT} sslmode=require"
)

pool = ConnectionPool(
    conninfo=CONNINFO,
    connection_class=OAuthConnection,
    min_size=1,
    max_size=10,
    open=True,
)

### Example query using the pool

### Only run the following cell where lakebase_type to provisioned

In [0]:
def query(sql: str, params=None):
    with pool.connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchall()

# Example: list tables in a schema
rows = query(
    """
    select * from "gap_fs".cc_static_features_online limit 5;
    """
)

for r in rows:
    print(r)


('ABF', 'MX', '100587271', '{"style_number": "589491", "clearance": false, "gender": "women", "color": "CHARCOAL", "category_ids": ["67852", "61997"], "sub_category_ids": ["67035", "59766", "59658", "15801", "79766", "26843", "80822"], "category_group": ["Accessories"], "category": ["Accessories|004", "Accessories|043", "Accessories|006"], "product_type": ["Sleepwear"], "style": ["Relaxed", "Fitted"], "department": ["Women"], "boutique": ["Weekend", "Everyday", "Date Night"], "marketing_flags": ["Trending", "Sale"], "collections": ["CoreFlex"]}', 'recs_feature_store', 'main', '1.2.0', datetime.date(2025, 7, 31))
('ABF', 'MX', '100587272', '{"style_number": "589710", "clearance": true, "gender": "unisex", "color": "LAVENDER", "category_ids": ["61121"], "sub_category_ids": ["89190", "34848", "23464", "58682", "27860"], "category_group": ["Dresses"], "category": ["Dresses|042", "Dresses|065"], "product_type": ["Loungewear"], "style": ["Oversized"], "department": ["Women"], "boutique": ["D

### Only run the following cell where lakebase_type to autoscaling

In [0]:
def query(sql: str, params=None):
    with pool.connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchall()

# Example: list tables in a schema
rows = query(
    """
    select * from public.t1;
    """
)

for r in rows:
    print(r)

(1,)
(2,)
(3,)
